[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkaitechnology-coder/free-python-for-development/blob/main/lesson-6-2-async-await-and-multithreading.ipynb)

# Lesson 6.2 · Async/Await & Multithreading

**Free Python For Development** · Topic 6: Advanced Python & Real-World Projects

_asyncio, aiohttp, threading, GIL._

▶ **Watch this lesson with the AI voice tutor:** [open it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-6-2)

**How to use this notebook:** run each code cell with **Shift + Enter**, or use **Runtime → Run all**. Then change the code and run it again — experimenting is the fastest way to learn.

**Setup** — install the libraries this lesson uses (takes a few seconds):

In [ ]:
!pip install -q aiohttp

## Introduction & Core Concepts of Async/Await & Multithreading

Have you ever wondered how Python programs can do many things at once — downloading multiple web pages or handling hundreds of requests in real time? Today we explore **async/await** and **multithreading**, two of the most powerful tools for making Python applications faster and more responsive.

Start with async and await. Imagine you are at a busy restaurant waiting for your food. Instead of standing still you chat with friends or check your phone while the kitchen works. Asynchronous programming lets your code do the same: rather than blocking while one task finishes, it can switch to another task.

> 💡 **Key insight:** `asyncio` is the coordinator (the “chef”) that schedules and resumes coroutines. An `async def` function is a coroutine; `await` is the point where it can politely pause and let other work run.

### Minimal Async Example

In [ ]:
import asyncio

async def fetch_data(name, delay):
    print(f"Start {name}")
    await asyncio.sleep(delay)   # non-blocking wait
    print(f"Done  {name}")
    return f"{name} result"

async def main():
    # Run three coroutines concurrently
    results = await asyncio.gather(
        fetch_data("A", 1.0),
        fetch_data("B", 0.5),
        fetch_data("C", 0.8),
    )
    print(results)

# A notebook already runs an event loop, so use top-level await here.
# In a normal .py script you would write: asyncio.run(main())
await main()

**What happens**

- `async def` creates a coroutine function.
- `await` suspends the current coroutine.
- The event loop runs other ready coroutines.
- `asyncio.gather` waits for all of them concurrently.

> 💡 **Mental model:** One thread, many cooperative tasks. While one task is waiting for I/O, the event loop gives the CPU to another task.

## Architecture & Implementation Deep Dive

Asyncio lets you write code that pauses and resumes, so web requests, database queries or file operations do not block everything else. When scraping several websites you can fetch them concurrently with `asyncio` + `aiohttp` instead of one after another.

Multithreading is different. If async is multitasking at a restaurant table, threading is having multiple chefs in the kitchen. Python’s `threading` module runs different parts of your code in separate OS threads. There is, however, an important limitation: the **Global Interpreter Lock (GIL)**.

**Async / asyncio**

Single thread, cooperative multitasking. Ideal for many concurrent I/O-bound operations. Low overhead, high concurrency.

**Threading**

Multiple OS threads. Useful for concurrent I/O or for integrating blocking libraries. Limited by the GIL for pure CPU work.

### Side-by-side Code

In [ ]:
import asyncio
import aiohttp
import threading
import time

# ---------- Async approach ----------
async def fetch(session, url):
    async with session.get(url) as resp:
        return await resp.text()

async def scrape_async(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, u) for u in urls]
        return await asyncio.gather(*tasks)

# ---------- Threading approach ----------
def worker(url, results, idx):
    # (blocking requests would go here)
    time.sleep(0.3)          # simulate I/O
    results[idx] = f"data from {url}"

def scrape_threaded(urls):
    results = [None] * len(urls)
    threads = []
    for i, url in enumerate(urls):
        t = threading.Thread(target=worker, args=(url, results, i))
        t.start()
        threads.append(t)
    for t in threads:
        t.join()
    return results

**Run both approaches** on a few real pages:

In [ ]:
urls = ["https://example.com", "https://www.python.org", "https://docs.python.org/3/"]

start = time.perf_counter()
pages = await scrape_async(urls)
print(f"async:   {len(pages)} pages in {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
results = scrape_threaded(urls)
print(f"threads: {results} in {time.perf_counter() - start:.2f}s")

**Key differences**

- Async: one thread, many tasks, explicit `await`.
- Threads: OS schedules them; shared memory needs locks.
- Async shines at high-concurrency I/O.
- Threads are convenient for wrapping blocking libraries.

## Best Practices, the GIL & When to Choose What

The GIL is like a single kitchen manager who allows only one chef to cook at a time. Even if you start many threads, only one can execute Python bytecode simultaneously. Consequently:

- Threading helps with I/O-bound work (the GIL is released during many blocking system calls).
- Threading does not speed up pure CPU-bound calculations; for that you need `multiprocessing` or native extensions.

### Decision Guide

**Prefer async when…**

You have many concurrent network or file operations (web scraping, API servers, chat bots, websocket handlers). You want high scalability with modest memory use.

**Prefer threading when…**

You must call blocking third-party libraries, need simple background workers, or the concurrency level is modest and code clarity matters more than extreme scale.

In [ ]:
import concurrent.futures
import multiprocessing
import time

def cpu_heavy(n):
    # pure Python CPU work — GIL bound
    total = 0
    for i in range(n):
        total += i * i
    return total

def io_bound(delay):
    time.sleep(delay)   # GIL is released
    return f"slept {delay}s"

# CPU-bound → use processes
# ("fork" lets worker processes use functions defined in this notebook)
start = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor(mp_context=multiprocessing.get_context("fork")) as pool:
    results = list(pool.map(cpu_heavy, [5_000_000] * 4))
print(f"CPU-bound with processes: {time.perf_counter() - start:.2f}s")

# I/O-bound → threads (or async) are fine
with concurrent.futures.ThreadPoolExecutor() as pool:
    results = list(pool.map(io_bound, [0.4, 0.3, 0.2, 0.1]))
print(results)

**Production habits**

- Measure first — don’t assume.
- Keep async code fully non-blocking.
- Protect shared state with locks when using threads.
- For heavy CPU work prefer `multiprocessing` or vectorized libraries.
- Combine: run async event loop and offload blocking calls to a thread pool.

> 💡 **Big takeaway:** Async/await gives you efficient concurrency for I/O-bound workloads on a single thread. Threading is useful for concurrent I/O and for wrapping blocking code, but the GIL limits pure-Python CPU scaling. Choose the right tool, experiment, and your applications will become faster, more responsive, and ready for real-world load.

---

✅ **Lesson complete!** Next up: **Lesson 6.3 · Web Scraping & Automation** — [watch it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-6-3) or open `lesson-6-3-web-scraping-and-automation.ipynb`.

Learn the full **Free Python For Development** course for free at [oratorlab.in](https://oratorlab.in/courses/python-for-development).